# Production RAG Lab: Vertex AI + Chroma

An end-to-end retrieval lab that goes beyond `chunking.ipynb`:

1. **Embeddings**: Google **Vertex AI** (`text-embedding-005`) instead of a local sentence-transformers model.
2. **Vector store**: a local, persistent **ChromaDB** instance — one collection per chunking strategy.
3. **Chunking bake-off**: index the same 3-document corpus with 9 chunking strategies and measure retrieval quality on a labeled query set.
4. **Production retrieval strategies**: metadata filtering, MMR, hybrid (dense + BM25), cross-encoder reranking, multi-query fusion, HyDE, and parent-document (small-to-big) retrieval — layered on top of the winning chunking strategy.
5. **RAG evaluation framework**: retrieval metrics (Recall@k, MRR) + generation metrics (faithfulness, answer relevance) via LLM-as-judge, wired into one comparison table.

This notebook assumes `chunking.ipynb` in this same folder for the conceptual explanation of each chunking strategy — here the focus is *measuring* which combination actually retrieves better.


## Prerequisites (read before running)

- A GCP project with the **Vertex AI API** enabled, and billing on (embeddings + Gemini calls in this notebook cost a small amount — a few cents for the whole notebook).
- Local auth: run `gcloud auth application-default login` in a terminal (not in this notebook), or set `GOOGLE_APPLICATION_CREDENTIALS` to a service-account JSON key.
- Set `PROJECT_ID` in the cell below to your actual project ID.

Nothing else in this notebook needs external services — Chroma persists to a local `./chroma_db` folder next to this notebook.


In [1]:
%pip install -q google-cloud-aiplatform langchain-google-vertexai
%pip install -q chromadb
%pip install -q langchain-text-splitters langchain-experimental tiktoken nltk scikit-learn sentence-transformers
%pip install -q rank_bm25 pandas


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
logfire 4.37.0 requires opentelemetry-sdk<1.43.0,>=1.39.0, but you have opentelemetry-sdk 1.44.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.42.1 requires opentelemetry-exporter-otlp-proto-common==1.42.1, but you have opentelemetry-exporter-otlp-proto-common 1.44.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.42.1 requires opentelemetry-proto==1.42.1, but you have opentelemetry-proto 1.44.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.42.1 requires opentelemetry-sdk~=1.42.1, but you have opentelemetry-sdk 1.44.0 which is incompatible.
opentelemetry-instrumentation 0.63b1 requires opentelemetry-semantic-conventions==0.63b1, but you have opentelemetry-semantic-conventions 0.65b0 which is incompatible.
opentelemetry-instrumentation-asgi 0.63b1 requires opentel

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 1. Vertex AI Setup

`embed_texts()` wraps Vertex AI's embedding model with batching (the API caps how many texts you can send per call) and a `task_type` — Vertex AI embeddings are trained with asymmetric query/document objectives, so **queries and documents must be embedded with different task types** (`RETRIEVAL_QUERY` vs `RETRIEVAL_DOCUMENT`) for best retrieval quality. This is a real production gotcha: embedding both sides the same way silently degrades ranking.


In [ ]:
import vertexai
from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput
from vertexai.generative_models import GenerativeModel

PROJECT_ID = "your-gcp-project-id"  # TODO: replace with your GCP project id
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")
generative_model = GenerativeModel("gemini-2.0-flash-001")

EMBED_BATCH_SIZE = 20  # stay comfortably under Vertex AI's per-request batch limit

def embed_texts(texts: list[str], task_type: str = "RETRIEVAL_DOCUMENT") -> list[list[float]]:
    """Embed a list of texts with Vertex AI, batching to respect API limits."""
    all_vecs: list[list[float]] = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = texts[i:i + EMBED_BATCH_SIZE]
        inputs = [TextEmbeddingInput(t, task_type) for t in batch]
        embeddings = embedding_model.get_embeddings(inputs)
        all_vecs.extend(e.values for e in embeddings)
    return all_vecs


In [ ]:
# Sanity check — confirms auth + project + API access are all working before we do real work.
try:
    _test_vec = embed_texts(["hello from vertex ai"])
    print(f"Vertex AI embeddings OK — dim={len(_test_vec[0])}")
    _test_reply = generative_model.generate_content("Reply with exactly one word: OK")
    print(f"Gemini OK — replied: {_test_reply.text.strip()!r}")
except Exception as exc:
    print("Vertex AI setup failed. Checklist:")
    print("  1. PROJECT_ID above is set to a real project")
    print("  2. `gcloud auth application-default login` has been run")
    print("  3. The Vertex AI API is enabled on that project")
    raise


## 2. ChromaDB Setup

One persistent collection per chunking strategy, so we can compare them side by side without re-embedding. `build_collection` always drops and recreates the collection so re-running a cell doesn't silently duplicate data.


In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

def build_collection(name: str, texts: list[str], metadatas: list[dict]) -> "chromadb.Collection":
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    collection = chroma_client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    ids = [f"{name}-{i}" for i in range(len(texts))]
    vectors = embed_texts(texts, task_type="RETRIEVAL_DOCUMENT")
    collection.add(ids=ids, documents=texts, metadatas=metadatas, embeddings=vectors)
    return collection

def query_collection(collection, query: str, n_results: int = 3, where: dict | None = None):
    query_vec = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    return collection.query(
        query_embeddings=[query_vec],
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"],
    )


## 3. Corpus

Three small documents spanning different domains, so retrieval queries have an unambiguous correct source to check against:

- `sample_document.md` — RAG concepts guide (from `chunking.ipynb`)
- `sample_code.py` — a small Python module (vector store + cosine similarity)
- `sample_faq.md` — a support/billing FAQ (new — gives us a non-technical domain)


In [ ]:
from pathlib import Path

CORPUS_PATHS = [Path("sample_document.md"), Path("sample_code.py"), Path("sample_faq.md")]
corpus = {p.name: p.read_text(encoding="utf-8") for p in CORPUS_PATHS}

for name, text in corpus.items():
    print(f"{name}: {len(text)} chars")


## 4. Chunking Strategies Under Test

Nine strategies, each implemented as `fn(path, text) -> list[str]` so they can be run uniformly over every document in the corpus. `unstructured`, LlamaIndex node parsers, and agentic/LLM chunking are intentionally left out here — they're covered standalone in `chunking.ipynb`, and for this comparison they'd either duplicate semantic/sentence chunking's behavior or add LLM cost per document without a new idea to test. Drop them into `CHUNK_STRATEGIES` below the same way if you want them in the bake-off.


In [ ]:
import re
import nltk
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    Language,
)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_vertexai import VertexAIEmbeddings
from sklearn.cluster import AgglomerativeClustering
from nltk.tokenize import sent_tokenize

nltk.download("punkt_tab", quiet=True)

lc_vertex_embeddings = VertexAIEmbeddings(model_name="text-embedding-005", project=PROJECT_ID, location=LOCATION)


def fixed_chunks(path: Path, text: str, size: int = 400) -> list[str]:
    return [c for c in (text[i:i + size] for i in range(0, len(text), size)) if c.strip()]


def recursive_chunks(path: Path, text: str) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
    return splitter.split_text(text)


def token_chunks(path: Path, text: str) -> list[str]:
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base", chunk_size=100, chunk_overlap=15
    )
    return splitter.split_text(text)


def sentence_chunks(path: Path, text: str, max_chars: int = 400) -> list[str]:
    prose = re.sub(r"[#*`]", "", text)
    sentences = sent_tokenize(prose)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current = f"{current} {sent}".strip()
        else:
            if current:
                chunks.append(current)
            current = sent
    if current:
        chunks.append(current)
    return chunks


def sliding_window_chunks(path: Path, text: str, window: int = 400, overlap: int = 100) -> list[str]:
    stride = window - overlap
    return [c for c in (text[i:i + window] for i in range(0, len(text), stride)) if c.strip()]


def markdown_header_chunks(path: Path, text: str) -> list[str]:
    if path.suffix not in (".md", ".markdown"):
        return recursive_chunks(path, text)
    sections = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")]
    ).split_text(text)
    sub_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
    return [c for sec in sections for c in sub_splitter.split_text(sec.page_content)]


def semantic_chunks(path: Path, text: str) -> list[str]:
    prose = re.sub(r"[#*`]|```.*?```", "", text, flags=re.S)
    splitter = SemanticChunker(
        lc_vertex_embeddings, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=80
    )
    return splitter.split_text(prose)


def cluster_chunks(path: Path, text: str) -> list[str]:
    prose = re.sub(r"^#.*$", "", text, flags=re.M)
    sentences = [s for s in sent_tokenize(prose) if s.strip()]
    if len(sentences) < 3:
        return [prose.strip()] if prose.strip() else []
    vecs = embed_texts(sentences, task_type="RETRIEVAL_DOCUMENT")
    labels = AgglomerativeClustering(
        n_clusters=None, distance_threshold=0.6, metric="cosine", linkage="average"
    ).fit_predict(vecs)
    chunks, current_label, current = [], labels[0], [sentences[0]]
    for label, sent in zip(labels[1:], sentences[1:]):
        if label == current_label:
            current.append(sent)
        else:
            chunks.append(" ".join(current))
            current_label, current = label, [sent]
    chunks.append(" ".join(current))
    return chunks


def document_aware_chunks(path: Path, text: str) -> list[str]:
    if path.suffix == ".py":
        splitter = RecursiveCharacterTextSplitter.from_language(Language.PYTHON, chunk_size=250, chunk_overlap=30)
        return splitter.split_text(text)
    if path.suffix in (".md", ".markdown"):
        return markdown_header_chunks(path, text)
    return recursive_chunks(path, text)


CHUNK_STRATEGIES = {
    "fixed": fixed_chunks,
    "recursive": recursive_chunks,
    "token": token_chunks,
    "sentence": sentence_chunks,
    "sliding_window": sliding_window_chunks,
    "markdown_header": markdown_header_chunks,
    "semantic": semantic_chunks,
    "cluster": cluster_chunks,
    "document_aware": document_aware_chunks,
}
print(f"{len(CHUNK_STRATEGIES)} strategies registered: {list(CHUNK_STRATEGIES)}")


In [ ]:
# Run every strategy over every document; attach uniform metadata for indexing.
all_chunks: dict[str, list[dict]] = {}

for strategy_name, strategy_fn in CHUNK_STRATEGIES.items():
    records = []
    for path in CORPUS_PATHS:
        text = corpus[path.name]
        chunks = strategy_fn(path, text)
        for i, chunk in enumerate(chunks):
            records.append({"text": chunk, "metadata": {"source": path.name, "strategy": strategy_name, "chunk_index": i}})
    all_chunks[strategy_name] = records
    print(f"{strategy_name:16s} -> {len(records):3d} chunks")


## 5. Indexing

Embed and index every strategy's chunks into its own Chroma collection (`chunks_<strategy>`). This is the expensive cell — it makes one batched Vertex AI embedding call per ~20 chunks, per strategy.


In [ ]:
import time

collections, index_timing_s = {}, {}

for strategy_name, records in all_chunks.items():
    texts = [r["text"] for r in records]
    metadatas = [r["metadata"] for r in records]
    start = time.perf_counter()
    collections[strategy_name] = build_collection(f"chunks_{strategy_name}", texts, metadatas)
    index_timing_s[strategy_name] = time.perf_counter() - start
    print(f"{strategy_name:16s} indexed {len(texts):3d} chunks in {index_timing_s[strategy_name]:.1f}s")


## 6. Labeled Eval Set

A small set of queries with a known correct source document — enough to compute Recall@k and MRR objectively instead of eyeballing results. Real projects should grow this to 50-100+ queries pulled from actual user questions or support tickets.


In [ ]:
EVAL_QUERIES = [
    {"query": "How do I get a refund after my subscription renews?", "expected_source": "sample_faq.md"},
    {"query": "What happens to my data if I delete my account?", "expected_source": "sample_faq.md"},
    {"query": "What response time can I expect from support on a paid plan?", "expected_source": "sample_faq.md"},
    {"query": "What does the VectorStore.search method return?", "expected_source": "sample_code.py"},
    {"query": "How is cosine similarity computed in this codebase?", "expected_source": "sample_code.py"},
    {"query": "Why does hybrid search often outperform pure vector search?", "expected_source": "sample_document.md"},
    {"query": "What is the lost-in-the-middle effect and why does it matter for prompting?", "expected_source": "sample_document.md"},
    {"query": "What two layers should RAG evaluation cover?", "expected_source": "sample_document.md"},
]
print(f"{len(EVAL_QUERIES)} eval queries")


## 7. Retrieval Metrics

- **Recall@k** — did the expected source appear anywhere in the top k results?
- **MRR** (Mean Reciprocal Rank) — `1 / rank` of the first correct result, averaged over queries (rewards ranking the right answer *higher*, not just anywhere in top k).

`evaluate_retrieval` takes a `retrieve_fn(query) -> list[source_str]` (ordered, most relevant first) so every retrieval strategy below — dense, hybrid, reranked, whatever — can be scored with the same function.


In [ ]:
import pandas as pd

def evaluate_retrieval(retrieve_fn, queries: list[dict] = EVAL_QUERIES, k: int = 3) -> dict:
    hits_at_1, hits_at_k, reciprocal_ranks = [], [], []
    for item in queries:
        sources = retrieve_fn(item["query"])[:k]
        expected = item["expected_source"]
        hits_at_1.append(sources[0] == expected if sources else False)
        hits_at_k.append(expected in sources)
        rank = next((i + 1 for i, s in enumerate(sources) if s == expected), None)
        reciprocal_ranks.append(1 / rank if rank else 0.0)
    return {
        "recall@1": sum(hits_at_1) / len(queries),
        f"recall@{k}": sum(hits_at_k) / len(queries),
        "mrr": sum(reciprocal_ranks) / len(queries),
    }


In [ ]:
def dense_retrieve_sources(strategy_name: str, query: str, k: int = 3) -> list[str]:
    result = query_collection(collections[strategy_name], query, n_results=k)
    return [m["source"] for m in result["metadatas"][0]]

baseline_rows = []
for strategy_name in CHUNK_STRATEGIES:
    metrics = evaluate_retrieval(lambda q, s=strategy_name: dense_retrieve_sources(s, q))
    baseline_rows.append({"strategy": strategy_name, "n_chunks": len(all_chunks[strategy_name]),
                           "index_time_s": round(index_timing_s[strategy_name], 1), **metrics})

baseline_df = pd.DataFrame(baseline_rows).sort_values("mrr", ascending=False).reset_index(drop=True)
baseline_df


Whichever strategy tops this table on `mrr` becomes `BEST_STRATEGY` below, used as the chunking layer for every production retrieval technique in the next section — this isolates "does the retrieval trick help" from "does the chunking strategy help," which is easy to conflate if you change both at once.


In [ ]:
BEST_STRATEGY = baseline_df.iloc[0]["strategy"]
best_collection = collections[BEST_STRATEGY]
best_records = all_chunks[BEST_STRATEGY]
print(f"BEST_STRATEGY = {BEST_STRATEGY!r}")


## 8. Production Retrieval Strategies

Everything below builds on `BEST_STRATEGY`'s collection. Each technique is a real production lever for improving retrieval beyond naive top-k dense search.


### 8a. Metadata Filtering

Restrict the candidate pool with Chroma's native `where` clause *before* similarity ranking runs — e.g. only search within one source document. Cheap, exact, and composes with every other technique below (you can filter, then MMR/rerank/fuse on top of the filtered pool).


In [ ]:
query = "what's covered under the refund policy"

print("Unfiltered (searches all 3 docs):")
for doc, meta, dist in zip(*[query_collection(best_collection, query, n_results=3)[k][0] for k in ("documents", "metadatas", "distances")]):
    print(f"  dist={dist:.3f} | {meta['source']} -> {doc[:80]}...")

print("\nFiltered to source == 'sample_faq.md':")
filtered = query_collection(best_collection, query, n_results=3, where={"source": "sample_faq.md"})
for doc, meta, dist in zip(filtered["documents"][0], filtered["metadatas"][0], filtered["distances"][0]):
    print(f"  dist={dist:.3f} | {meta['source']} -> {doc[:80]}...")


### 8b. MMR (Maximal Marginal Relevance)

Plain top-k similarity search can return k near-duplicate chunks that all restate the same sentence. MMR pulls a larger candidate pool, then greedily selects results that balance **relevance to the query** against **dissimilarity to what's already been picked** — controlled by `lambda_mult` (1.0 = pure relevance, 0.0 = pure diversity).


In [ ]:
import numpy as np

def mmr_rerank(query_vec, candidate_vecs, candidate_docs, candidate_meta, k=3, lambda_mult=0.5):
    query_vec = np.array(query_vec)
    candidate_vecs = np.array(candidate_vecs)
    sims_to_query = candidate_vecs @ query_vec / (np.linalg.norm(candidate_vecs, axis=1) * np.linalg.norm(query_vec) + 1e-9)

    selected, remaining = [], list(range(len(candidate_docs)))
    while remaining and len(selected) < k:
        if not selected:
            best = max(remaining, key=lambda i: sims_to_query[i])
        else:
            def mmr_score(i):
                redundancy = max(
                    candidate_vecs[i] @ candidate_vecs[j] / (np.linalg.norm(candidate_vecs[i]) * np.linalg.norm(candidate_vecs[j]) + 1e-9)
                    for j in selected
                )
                return lambda_mult * sims_to_query[i] - (1 - lambda_mult) * redundancy
            best = max(remaining, key=mmr_score)
        selected.append(best)
        remaining.remove(best)
    return [candidate_docs[i] for i in selected], [candidate_meta[i] for i in selected]


def mmr_retrieve_sources(query: str, k: int = 3, fetch_k: int = 12, lambda_mult: float = 0.5) -> list[str]:
    query_vec = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    candidates = best_collection.query(query_embeddings=[query_vec], n_results=fetch_k, include=["documents", "metadatas", "embeddings"])
    docs, metas = mmr_rerank(query_vec, candidates["embeddings"][0], candidates["documents"][0], candidates["metadatas"][0], k=k, lambda_mult=lambda_mult)
    return [m["source"] for m in metas]

mmr_metrics = evaluate_retrieval(mmr_retrieve_sources)
mmr_metrics


### 8c. Hybrid Search (Dense + BM25)

Dense embeddings are good at paraphrase/meaning, weak on exact rare tokens (error codes, function names, product SKUs). BM25 is the opposite. Combine their rankings with **Reciprocal Rank Fusion** (`score = sum(1 / (rank + c))` across each ranking) — simple, and doesn't require normalizing two incompatible score scales.

```
pip install rank_bm25
```


In [ ]:
from rank_bm25 import BM25Okapi

best_texts = [r["text"] for r in best_records]
best_metas = [r["metadata"] for r in best_records]
bm25 = BM25Okapi([t.lower().split() for t in best_texts])

def reciprocal_rank_fusion(rankings: list[list[int]], k: int = 60) -> list[int]:
    scores: dict[int, float] = {}
    for ranking in rankings:
        for rank, idx in enumerate(ranking):
            scores[idx] = scores.get(idx, 0.0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_retrieve_sources(query: str, k: int = 3, fetch_k: int = 10) -> list[str]:
    query_vec = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    dense = best_collection.query(query_embeddings=[query_vec], n_results=fetch_k, include=["documents"])
    dense_ranking = [best_texts.index(d) for d in dense["documents"][0] if d in best_texts]

    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranking = list(np.argsort(bm25_scores)[::-1][:fetch_k])

    fused = reciprocal_rank_fusion([dense_ranking, bm25_ranking])[:k]
    return [best_metas[i]["source"] for i in fused]

hybrid_metrics = evaluate_retrieval(hybrid_retrieve_sources)
hybrid_metrics


### 8d. Cross-Encoder Reranking

Dense retrieval scores query and document independently (fast, approximate). A cross-encoder scores the (query, document) *pair* jointly through a full transformer forward pass (slow, but far more accurate) — used as a second stage to rerank a small candidate set, not to search the whole index. This is the same shape as a production setup using a managed reranker (e.g. Vertex AI's Ranking API / Discovery Engine, or Cohere Rerank) instead of a local model.

```
pip install sentence-transformers
```


In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_retrieve_sources(query: str, k: int = 3, fetch_k: int = 10) -> list[str]:
    candidates = query_collection(best_collection, query, n_results=fetch_k)
    docs, metas = candidates["documents"][0], candidates["metadatas"][0]
    if not docs:
        return []
    pair_scores = cross_encoder.predict([(query, d) for d in docs])
    ranked = sorted(zip(pair_scores, metas), key=lambda pair: pair[0], reverse=True)
    return [m["source"] for _, m in ranked[:k]]

rerank_metrics = evaluate_retrieval(rerank_retrieve_sources)
rerank_metrics


### 8e. Multi-Query Retrieval (Query Expansion)

A single query is one phrasing of an information need — if the corpus uses different words for the same concept, dense search can miss it. An LLM generates a few alternate phrasings, each is retrieved separately, and the rankings are fused (again with RRF). This trades extra latency/cost for recall on queries whose vocabulary doesn't match the source text.


In [ ]:
import json

MULTI_QUERY_PROMPT = """Generate 3 different rephrasings of this search query that preserve its meaning \
but vary the vocabulary/phrasing, to widen a semantic search. Return ONLY a JSON array of 3 strings.

QUERY: {query}"""

def expand_query(query: str) -> list[str]:
    response = generative_model.generate_content(MULTI_QUERY_PROMPT.format(query=query))
    raw = response.text.strip().strip("`").removeprefix("json").strip()
    return json.loads(raw)

def multi_query_retrieve_sources(query: str, k: int = 3, fetch_k: int = 5) -> list[str]:
    variants = [query] + expand_query(query)
    rankings = []
    for variant in variants:
        result = query_collection(best_collection, variant, n_results=fetch_k)
        rankings.append([best_texts.index(d) for d in result["documents"][0] if d in best_texts])
    fused = reciprocal_rank_fusion(rankings)[:k]
    return [best_metas[i]["source"] for i in fused]

multi_query_metrics = evaluate_retrieval(multi_query_retrieve_sources)
multi_query_metrics


### 8f. HyDE (Hypothetical Document Embeddings)

Queries are short questions; chunks are longer declarative passages — that asymmetry costs some similarity even when the chunk fully answers the query. HyDE asks an LLM to *hallucinate* a plausible answer passage first, then embeds **that** (with `RETRIEVAL_DOCUMENT`, since it now reads like a document, not a query) instead of the raw query. The hypothetical answer doesn't need to be factually correct — it just needs to be *stylistically and topically* closer to the real chunks than the bare question was.


In [ ]:
HYDE_PROMPT = """Write a short, plausible-sounding passage (3-4 sentences) that would answer this question, \
as if it came directly from the source documentation. Don't hedge or say you're not sure — just write the passage.

QUESTION: {query}"""

def hyde_retrieve_sources(query: str, k: int = 3) -> list[str]:
    hypothetical = generative_model.generate_content(HYDE_PROMPT.format(query=query)).text.strip()
    hyde_vec = embed_texts([hypothetical], task_type="RETRIEVAL_DOCUMENT")[0]
    result = best_collection.query(query_embeddings=[hyde_vec], n_results=k, include=["metadatas"])
    return [m["source"] for m in result["metadatas"][0]]

hyde_metrics = evaluate_retrieval(hyde_retrieve_sources)
hyde_metrics


### 8g. Parent-Document (Small-to-Big) Retrieval

Small chunks retrieve precisely (less irrelevant text diluting the embedding) but read poorly as standalone context (missing surroundings). This pattern indexes small **child** chunks for search, but each child carries a `parent_id` pointing back to its full **parent** section — so retrieval precision comes from the child, but the context handed to the LLM is the full parent.


In [ ]:
parent_lookup: dict[str, str] = {}
child_records = []

for path in CORPUS_PATHS:
    text = corpus[path.name]
    parents = markdown_header_chunks(path, text) if path.suffix in (".md", ".markdown") else recursive_chunks(path, text)
    for p_idx, parent_text in enumerate(parents):
        parent_id = f"{path.name}-p{p_idx}"
        parent_lookup[parent_id] = parent_text
        children = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=0).split_text(parent_text)
        for c_idx, child_text in enumerate(children):
            child_records.append({
                "text": child_text,
                "metadata": {"source": path.name, "parent_id": parent_id, "chunk_index": c_idx},
            })

parent_child_collection = build_collection(
    "chunks_parent_child",
    [r["text"] for r in child_records],
    [r["metadata"] for r in child_records],
)
print(f"{len(parent_lookup)} parents -> {len(child_records)} child chunks indexed")

def parent_document_retrieve(query: str, k: int = 3) -> list[dict]:
    """Returns full parent context (not just the matched child) for each hit."""
    result = query_collection(parent_child_collection, query, n_results=k)
    seen_parents, hits = set(), []
    for meta in result["metadatas"][0]:
        if meta["parent_id"] not in seen_parents:
            seen_parents.add(meta["parent_id"])
            hits.append({"source": meta["source"], "text": parent_lookup[meta["parent_id"]]})
    return hits

def parent_document_retrieve_sources(query: str, k: int = 3) -> list[str]:
    return [h["source"] for h in parent_document_retrieve(query, k=k)]

parent_doc_metrics = evaluate_retrieval(parent_document_retrieve_sources)
parent_doc_metrics


## 9. Retrieval Strategy Comparison

All techniques above, evaluated on the same query set, layered on `BEST_STRATEGY`'s chunks.


In [ ]:
retrieval_rows = [
    {"technique": f"dense only ({BEST_STRATEGY})", **evaluate_retrieval(lambda q: dense_retrieve_sources(BEST_STRATEGY, q))},
    {"technique": "dense + MMR", **mmr_metrics},
    {"technique": "hybrid (dense + BM25)", **hybrid_metrics},
    {"technique": "dense + cross-encoder rerank", **rerank_metrics},
    {"technique": "multi-query fusion", **multi_query_metrics},
    {"technique": "HyDE", **hyde_metrics},
    {"technique": "parent-document (small-to-big)", **parent_doc_metrics},
]
retrieval_df = pd.DataFrame(retrieval_rows).sort_values("mrr", ascending=False).reset_index(drop=True)
retrieval_df


## 10. RAG Evaluation Framework

Retrieval metrics (above) only tell you whether the *right chunks* were fetched. They say nothing about whether the *generated answer* actually uses them correctly. Production RAG eval needs both layers:

- **Faithfulness** — is every claim in the answer supported by the retrieved context? (catches hallucination even when retrieval was correct)
- **Answer Relevance** — does the answer actually address the question asked? (catches correct-but-off-topic answers)

Both are scored here with an LLM-as-judge (Gemini) on a 1-5 rubric. For a larger production eval suite, consider the [`ragas`](https://github.com/explodinggradients/ragas) library, which implements these same metrics plus context precision/recall — it defaults to OpenAI but can be pointed at Vertex AI via `ragas.llms.LangchainLLMWrapper(ChatVertexAI(...))`.


In [ ]:
RAG_ANSWER_PROMPT = """Answer the question using ONLY the context below. If the context doesn't contain the \
answer, say so explicitly rather than guessing.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

FAITHFULNESS_PROMPT = """Rate how well the ANSWER is supported by the CONTEXT, from 1 (unsupported / \
contradicts context) to 5 (every claim is directly supported). Return ONLY JSON: {{"score": <1-5>, "rationale": "<one sentence>"}}

CONTEXT:
{context}

ANSWER:
{answer}"""

RELEVANCE_PROMPT = """Rate how directly the ANSWER addresses the QUESTION, from 1 (off-topic / non-answer) \
to 5 (fully addresses it). Return ONLY JSON: {{"score": <1-5>, "rationale": "<one sentence>"}}

QUESTION: {query}

ANSWER:
{answer}"""

def _judge(prompt: str) -> dict:
    raw = generative_model.generate_content(prompt).text.strip().strip("`").removeprefix("json").strip()
    return json.loads(raw)

def generate_answer(query: str, context_chunks: list[str]) -> str:
    context = "\n\n".join(f"[{i}] {c}" for i, c in enumerate(context_chunks))
    return generative_model.generate_content(RAG_ANSWER_PROMPT.format(context=context, query=query)).text.strip()

def judge_faithfulness(answer: str, context_chunks: list[str]) -> dict:
    return _judge(FAITHFULNESS_PROMPT.format(context="\n\n".join(context_chunks), answer=answer))

def judge_relevance(answer: str, query: str) -> dict:
    return _judge(RELEVANCE_PROMPT.format(query=query, answer=answer))


In [ ]:
def evaluate_rag_pipeline(retrieve_texts_fn, queries: list[dict] = EVAL_QUERIES, k: int = 3) -> pd.DataFrame:
    """retrieve_texts_fn(query, k) -> list[str] of chunk texts (not just sources)."""
    rows = []
    for item in queries:
        start = time.perf_counter()
        context_chunks = retrieve_texts_fn(item["query"], k)
        answer = generate_answer(item["query"], context_chunks)
        faithfulness = judge_faithfulness(answer, context_chunks)
        relevance = judge_relevance(answer, item["query"])
        rows.append({
            "query": item["query"],
            "faithfulness": faithfulness["score"],
            "relevance": relevance["score"],
            "latency_s": round(time.perf_counter() - start, 1),
        })
    return pd.DataFrame(rows)


def dense_retrieve_texts(query: str, k: int = 3) -> list[str]:
    result = query_collection(best_collection, query, n_results=k)
    return result["documents"][0]

dense_rag_results = evaluate_rag_pipeline(dense_retrieve_texts)
dense_rag_results


## 11. Final Comparison — Retrieval + Generation Together

Run the full pipeline (retrieve -> generate -> judge) for a shortlist of configs: naive dense (baseline), the best-performing production retrieval technique from section 9, and parent-document retrieval (since it changes *what* gets passed to generation, not just what gets ranked). This is the table that actually answers "which setup should we ship."


In [ ]:
def hybrid_retrieve_texts(query: str, k: int = 3, fetch_k: int = 10) -> list[str]:
    query_vec = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    dense = best_collection.query(query_embeddings=[query_vec], n_results=fetch_k, include=["documents"])
    dense_ranking = [best_texts.index(d) for d in dense["documents"][0] if d in best_texts]
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranking = list(np.argsort(bm25_scores)[::-1][:fetch_k])
    fused = reciprocal_rank_fusion([dense_ranking, bm25_ranking])[:k]
    return [best_texts[i] for i in fused]


def parent_document_retrieve_texts(query: str, k: int = 3) -> list[str]:
    return [h["text"] for h in parent_document_retrieve(query, k=k)]


final_configs = {
    "dense only": dense_retrieve_texts,
    "hybrid (dense + BM25)": hybrid_retrieve_texts,
    "parent-document": parent_document_retrieve_texts,
}

final_rows = []
for name, retrieve_fn in final_configs.items():
    rag_results = evaluate_rag_pipeline(retrieve_fn)
    retrieval_metrics = evaluate_retrieval(
        parent_document_retrieve_sources if name == "parent-document"
        else (hybrid_retrieve_sources if name.startswith("hybrid") else (lambda q: dense_retrieve_sources(BEST_STRATEGY, q)))
    )
    final_rows.append({
        "config": name,
        **retrieval_metrics,
        "avg_faithfulness": round(rag_results["faithfulness"].mean(), 2),
        "avg_relevance": round(rag_results["relevance"].mean(), 2),
        "avg_latency_s": round(rag_results["latency_s"].mean(), 1),
    })

final_df = pd.DataFrame(final_rows).sort_values("mrr", ascending=False).reset_index(drop=True)
final_df


## Summary

- **Chunking** sets the ceiling on retrieval quality — no retrieval trick fixes chunks that split the answer from the question. Section 7's `baseline_df` tells you which strategy to start from for *this* corpus; re-run it whenever the corpus composition changes meaningfully.
- **Retrieval technique** is where most of the remaining gains come from without re-chunking: hybrid search generally helps most on queries with exact terms (names, codes), MMR helps when top-k results are redundant, reranking helps when the embedding model's coarse ranking needs a precision pass, multi-query/HyDE help when query and document vocabulary diverge, and parent-document retrieval helps when small precise chunks read poorly out of context.
- **Faithfulness and relevance** are a separate axis from retrieval metrics — a config can have great Recall@k and still generate poorly-grounded answers if the prompt or model is weak, and vice versa. Track both, always on the same query set, so results are directly comparable — never eyeball retrieval and generation quality on different query sets and assume the conclusions transfer.
- For production scale: grow `EVAL_QUERIES` to 50-100+ real queries (ideally mined from actual usage/support tickets), and consider `ragas` or Vertex AI's managed **Gen AI Evaluation Service** once this custom harness is validated against it.
